## ÉTAPE 14 — Séries temporelles
Objectif : analyser et prévoir les tendances
pip install statsmodels


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller

df = pd.read_csv("JEU_DE_DONNEE.csv")

df_ble = df[
    (df["category"] == "wheat") &
    (df["country_or_area"] == "World +") &
    (df["element"] == "Production Quantity")
].dropna(subset=["value"]).sort_values("year")

serie = pd.Series(
    df_ble["value"].values / 1e6,
    index=pd.to_datetime(df_ble["year"].astype(int), format="%Y")
)

print(f"Série temporelle : {len(serie)} points (1961–2007)")


## 1. TEST DE STATIONNARITÉ (Dickey-Fuller)


## Une série stationnaire = moyenne et variance stables
Nécessaire pour ARIMA


In [ ]:
resultat = adfuller(serie)
print(f"\nTest Dickey-Fuller :")
print(f"  p-value = {resultat[1]:.4f}")
print(f"  → {'Série stationnaire ✓' if resultat[1] < 0.05 else 'Série NON stationnaire (tendance présente)'}")


## 2. DÉCOMPOSITION — TENDANCE + SAISONNALITÉ


In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomp = seasonal_decompose(serie, model="additive", period=5)

fig, axes = plt.subplots(4, 1, figsize=(13, 10))
serie.plot(ax=axes[0], title="Série originale", color="#2563EB")
decomp.trend.plot(ax=axes[1], title="Tendance", color="#DC2626")
decomp.seasonal.plot(ax=axes[2], title="Saisonnalité", color="#16A34A")
decomp.resid.plot(ax=axes[3], title="Résidus", color="#D97706")

for ax in axes:
    ax.grid(alpha=0.3)
    ax.set_xlabel("")

plt.suptitle("Décomposition de la série temporelle — Blé mondial",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("decomposition_serie.png", dpi=150, bbox_inches="tight")
print("\nDécomposition sauvegardée ✓")
plt.show()


## 3. LISSAGE EXPONENTIEL (Holt-Winters)


## Modèle simple mais efficace pour les prévisions


In [ ]:
modele = ExponentialSmoothing(serie, trend="add", seasonal=None)
fit = modele.fit()

# Prévision sur 10 ans
previsions = fit.forecast(10)

fig2, ax = plt.subplots(figsize=(13, 6))

serie.plot(ax=ax, color="#2563EB", linewidth=2, label="Données réelles")
fit.fittedvalues.plot(ax=ax, color="orange", linewidth=1.5,
                      linestyle="--", label="Valeurs lissées")
previsions.plot(ax=ax, color="red", linewidth=2.5,
                linestyle="--", marker="o", markersize=5, label="Prévisions 2008–2017")

ax.axvline(pd.Timestamp("2007"), color="gray", linestyle=":", alpha=0.7)
ax.set_title("Prévision — Production mondiale de blé (Holt-Winters)",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Année")
ax.set_ylabel("Production (millions de tonnes)")
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("prevision_ble.png", dpi=150, bbox_inches="tight")
print("Prévisions sauvegardées ✓")
plt.show()

print("\n--- Prévisions 2008–2017 ---")
for annee, valeur in zip(range(2008, 2018), previsions):
    print(f"  {annee} : {valeur:.0f} Mt prévus")

print("\n=== RÉSUMÉ SÉRIES TEMPORELLES ===")
print("seasonal_decompose()       → tendance + saisonnalité + résidus")
print("adfuller()                 → tester la stationnarité")
print("ExponentialSmoothing()     → modèle de prévision simple")
print(".forecast(n)               → prévoir n périodes futures")
